In [74]:
# [MERGE 1] Load main + modern full datasets and align schemas

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset")

main_path = BASE_DIR / "data" / "final" / "publication" / "idiomx_full.parquet"
modern_path = BASE_DIR / "data" / "final" / "publication" / "idiomx_modern_full.parquet"

output_parquet = BASE_DIR / "data" / "final" / "publication" / "idiomx_extended_full.parquet"
output_csv = BASE_DIR / "data" / "final" / "publication" / "csv" / "idiomx_extended_full.csv"

df_main = pd.read_parquet(main_path)
df_modern = pd.read_parquet(modern_path)

print("Main shape  :", df_main.shape)
print("Modern shape:", df_modern.shape)

Main shape  : (174956, 52)
Modern shape: (14524, 63)


In [75]:
# [MODERN FIX 1] Inspect nulls in important modern metadata fields

important_cols = [
    "source",
    "source_type",
    "source_url",
    "record_origin",
    "license_source",
    "pos",
    "tags",
    "idiom_confidence",
    "enrichment_model",
    "enrichment_version",
    "is_example_idiom",
]

null_summary = (
    df_modern[important_cols]
    .isna()
    .sum()
    .to_frame("null_count")
)

null_summary["null_ratio"] = (null_summary["null_count"] / len(df_modern)).round(4)

print("Modern dataset shape:", df_modern.shape)
display(null_summary)

Modern dataset shape: (14524, 63)


,null_count,null_ratio
source,0,0.0000
source_type,0,0.0000
source_url,0,0.0000
record_origin,0,0.0000
license_source,0,0.0000
pos,0,0.0000
tags,0,0.0000
idiom_confidence,13001,0.8951
enrichment_model,0,0.0000
enrichment_version,0,0.0000


In [76]:
# [MODERN FIX 3A] Inspect column names in each cleaned source file

from pathlib import Path
import pandas as pd

processed_dir = BASE_DIR / "data" / "processed"

source_files = [
    processed_dir / "idioms_source_wiktionary_slang_cleaned.csv",
    processed_dir / "idioms_source_urban_dictionary_cleaned.csv",
    processed_dir / "idioms_source_opensubtitles_candidates_cleaned.csv",  # optional
]

for path in source_files:
    if path.exists():
        temp = pd.read_csv(path, nrows=3, low_memory=False)
        print(f"\n=== {path.name} ===")
        print(list(temp.columns))
        display(temp.head(2))
    else:
        print(f"\n=== {path.name} ===")
        print("File not found")


=== idioms_source_wiktionary_slang_cleaned.csv ===
['idiom', 'meaning_en', 'example', 'source', 'source_type', 'pos', 'tags', 'idiom_confidence', 'source_url']


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,$100 hamburger,A general aviation flight that involves flying...,It’s called hunting the $100 hamburger — “$100...,wiktionary_slang_kaikki,dictionary,noun,"slang,aeronautics,aerospace,aviation,business,...",medium,https://en.wiktionary.org/wiki/%24100%20hamburger
1,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,wiktionary_slang_kaikki,dictionary,phrase,informal,medium,https://en.wiktionary.org/wiki/%27ark%20at%20%...



=== idioms_source_urban_dictionary_cleaned.csv ===
['idiom', 'meaning_en', 'example', 'source', 'source_type', 'pos', 'tags', 'idiom_confidence', 'source_url']


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,a cap,A bullet,"If he run his mouth to me one more time , I'll...",urban_dictionary,crowdsourced_dictionary,NaN,"slang,colloquial,informal,modern",low,https://www.urbandictionary.com/define.php?ter...
1,a clapback,Another way of saying having a strong Rebuttal .,SEE WORKAHOLICS *( Actually mentions Clapback)...,urban_dictionary,crowdsourced_dictionary,NaN,"slang,colloquial,informal,modern",low,https://www.urbandictionary.com/define.php?ter...



=== idioms_source_opensubtitles_candidates_cleaned.csv ===
['idiom', 'meaning_en', 'example', 'source', 'source_type', 'pos', 'tags', 'idiom_confidence', 'source_url']


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,a and camille blot wellens,NaN,The reconstruction was carried out in 2002 by ...,opensubtitles_dialogue,subtitle_dialogue,NaN,"dialogue,conversational,candidate,weak_candida...",low,NaN
1,a bad feeling about,NaN,I have a bad feeling about this place.,opensubtitles_dialogue,subtitle_dialogue,NaN,"dialogue,conversational,candidate,weak_candida...",low,NaN


In [77]:
# [MODERN FIX 4] Fill null source metadata in df_modern from cleaned source files

# normalize join key in modern dataset
df_modern["idiom_canonical_norm"] = (
    df_modern["idiom_canonical"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# build source metadata table from cleaned files
source_dfs = []

for path in source_files:
    if path.exists():
        temp = pd.read_csv(path, low_memory=False)

        if "idiom" in temp.columns:
            temp = temp.rename(columns={"idiom": "idiom_canonical"})

        temp["idiom_canonical_norm"] = (
            temp["idiom_canonical"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        temp["__source_file"] = path.name
        source_dfs.append(temp)

df_source_meta = pd.concat(source_dfs, ignore_index=True)

# keep best available metadata columns
meta_cols = [
    "idiom_canonical_norm",
    "source",
    "source_type",
    "source_url",
    "pos",
    "tags",
    "idiom_confidence",
    "__source_file",
]

meta_cols = [c for c in meta_cols if c in df_source_meta.columns]
df_source_meta = df_source_meta[meta_cols].copy()

# if multiple rows per idiom exist, keep first non-null-rich row
df_source_meta["_non_null_count"] = df_source_meta.notna().sum(axis=1)
df_source_meta = (
    df_source_meta
    .sort_values(["idiom_canonical_norm", "_non_null_count"], ascending=[True, False])
    .drop_duplicates(subset=["idiom_canonical_norm"], keep="first")
    .drop(columns=["_non_null_count"])
    .copy()
)

# merge metadata into modern dataset
df_modern = df_modern.merge(
    df_source_meta,
    on="idiom_canonical_norm",
    how="left",
    suffixes=("", "_src")
)

# fill only nulls in target columns
fill_map = {
    "source": "source_src",
    "source_type": "source_type_src",
    "source_url": "source_url_src",
    "pos": "pos_src",
    "tags": "tags_src",
    "idiom_confidence": "idiom_confidence_src",
}

for target_col, src_col in fill_map.items():
    if target_col in df_modern.columns and src_col in df_modern.columns:
        df_modern[target_col] = df_modern[target_col].combine_first(df_modern[src_col])

# cleanup helper columns from merge
drop_cols = [c for c in df_modern.columns if c.endswith("_src")]
df_modern = df_modern.drop(columns=drop_cols, errors="ignore")

print("Null counts after filling source metadata:")
display(
    df_modern[
        ["source", "source_type", "source_url", "pos", "tags", "idiom_confidence"]
    ].isna().sum().to_frame("null_count")
)

Null counts after filling source metadata:


C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\2633290500.py:78: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df_modern[target_col] = df_modern[target_col].combine_first(df_modern[src_col])


,null_count
source,0
source_type,0
source_url,0
pos,0
tags,0
idiom_confidence,0


In [78]:
# [MODERN FIX 5] Fill remaining metadata nulls with controlled defaults

# --------------------------------------------------
# Source-related defaults
# --------------------------------------------------
df_modern["source"] = df_modern["source"].fillna("modern_combined")
df_modern["source_type"] = df_modern["source_type"].fillna("slang_collection")

# source_url may genuinely be missing
df_modern["source_url"] = df_modern["source_url"].fillna("unknown")

# POS / tags defaults
df_modern["pos"] = df_modern["pos"].fillna("phrase")
df_modern["tags"] = df_modern["tags"].fillna("idiomatic_expression")

# confidence after filtering/validation can safely be high by default
df_modern["idiom_confidence"] = df_modern["idiom_confidence"].fillna(1.0)

# --------------------------------------------------
# Provenance fields
# --------------------------------------------------
if "record_origin" not in df_modern.columns:
    df_modern["record_origin"] = pd.NA
df_modern["record_origin"] = df_modern["record_origin"].fillna("modern_pipeline")

if "license_source" not in df_modern.columns:
    df_modern["license_source"] = pd.NA
df_modern["license_source"] = df_modern["license_source"].fillna("mixed_sources")

# --------------------------------------------------
# Enrichment metadata
# --------------------------------------------------
if "enrichment_model" not in df_modern.columns:
    df_modern["enrichment_model"] = pd.NA
df_modern["enrichment_model"] = df_modern["enrichment_model"].fillna("gpt-4.1-mini")

if "enrichment_version" not in df_modern.columns:
    df_modern["enrichment_version"] = pd.NA
df_modern["enrichment_version"] = df_modern["enrichment_version"].fillna("modern_v1")

# --------------------------------------------------
# Fix is_example_idiom from label
# idiomatic -> True
# literal -> False
# borderline -> False
# --------------------------------------------------
if "is_example_idiom" in df_modern.columns and "example_usage_label" in df_modern.columns:
    mapped_flags = df_modern["example_usage_label"].map({
        "idiomatic": True,
        "literal": False,
        "borderline": False,
    })
    df_modern["is_example_idiom"] = df_modern["is_example_idiom"].combine_first(mapped_flags)

# --------------------------------------------------
# Inspect remaining nulls
# --------------------------------------------------
check_cols = [
    "source",
    "source_type",
    "source_url",
    "record_origin",
    "license_source",
    "pos",
    "tags",
    "idiom_confidence",
    "enrichment_model",
    "enrichment_version",
    "is_example_idiom",
]

print("Remaining null counts after controlled fill:")
display(df_modern[check_cols].isna().sum().to_frame("null_count"))

Remaining null counts after controlled fill:


C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\3567874252.py:53: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df_modern["is_example_idiom"] = df_modern["is_example_idiom"].combine_first(mapped_flags)


,null_count
source,0
source_type,0
source_url,0
record_origin,0
license_source,0
pos,0
tags,0
idiom_confidence,0
enrichment_model,0
enrichment_version,0


In [79]:
df_modern.columns

Index(['adversarial_type', 'ambiguity_flag', 'context_type',
       'enrichment_model', 'enrichment_version', 'example', 'example_language',
       'example_normalized', 'example_raw', 'example_usage_label',
       'expected_label', 'explanation_ar', 'explanation_en', 'explanation_fr',
       'hard_negative_idioms', 'idiom_canonical', 'idiom_canonical_meaning',
       'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french',
       'idiom_compositionality_level', 'idiom_confidence', 'idiom_domain',
       'idiom_id', 'idiom_in_example_arabic', 'idiom_in_example_french',
       'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_en',
       'idiom_in_example_meaning_french', 'idiom_level_explanation_ar',
       'idiom_level_explanation_en', 'idiom_level_explanation_fr',
       'idiom_register', 'idiom_surface', 'idiom_validity_label',
       'is_adversarial_example', 'is_example_idiom', 'is_generated_example',
       'is_idiom', 'learner_difficulty', 'license_source',

In [80]:
# Basic sanity checks
print("Total rows:", len(df_modern))
print("Unique idioms:", df_modern["idiom_canonical"].nunique())

print("\nLabel distribution:")
display(df_modern["example_usage_label"].value_counts(normalize=True))

Total rows: 14524
Unique idioms: 1515

Label distribution:


example_usage_label
idiomatic     0.404365
literal       0.379716
borderline    0.215918
Name: proportion, dtype: Float64

In [81]:
# [MODERN FIX FINAL] Drop helper columns before saving

drop_cols = ["__source_file"]

df_modern = df_modern.drop(columns=[c for c in drop_cols if c in df_modern.columns])

print("Remaining nulls after cleanup:")
display(df_modern.isna().sum()[df_modern.isna().sum() > 0].to_frame("null_count"))

Remaining nulls after cleanup:


,null_count
adversarial_type,11388
example_raw,14524
minimal_pair_id,3514


In [82]:
# [MODERN FINAL SAVE FIX] Clean dtypes before parquet save

import pandas as pd

# numeric columns
numeric_cols = [
    "idiom_confidence",
    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
]

for col in numeric_cols:
    if col in df_modern.columns:
        df_modern[col] = pd.to_numeric(df_modern[col], errors="coerce")

# boolean columns
bool_cols = [
    "is_example_idiom",
    "is_generated_example",
    "is_adversarial_example",
    "is_idiom",
]

for col in bool_cols:
    if col in df_modern.columns:
        df_modern[col] = df_modern[col].astype("boolean")

# string-like columns
for col in df_modern.columns:
    if col not in numeric_cols + bool_cols:
        df_modern[col] = df_modern[col].astype("string")

print(df_modern.dtypes)

adversarial_type        string[python]
ambiguity_flag          string[python]
context_type            string[python]
enrichment_model        string[python]
enrichment_version      string[python]
                             ...      
source_type             string[python]
source_url              string[python]
tags                    string[python]
validation_status       string[python]
idiom_canonical_norm    string[python]
Length: 64, dtype: object


In [83]:
# [MODERN FINAL SAVE] Save cleaned modern dataset

modern_full_parquet_clean = BASE_DIR / "data" / "final" / "publication" / "idiomx_modern_full.parquet"
modern_full_csv_clean = BASE_DIR / "data" / "final" / "publication" / "csv" / "idiomx_modern_full.csv"

modern_full_parquet_clean.parent.mkdir(parents=True, exist_ok=True)
modern_full_csv_clean.parent.mkdir(parents=True, exist_ok=True)

df_modern.to_parquet(modern_full_parquet_clean, index=False)
df_modern.to_csv(modern_full_csv_clean, index=False, encoding="utf-8-sig")

print("Saved modern full parquet:", modern_full_parquet_clean)
print("Saved modern full csv    :", modern_full_csv_clean)
print("Shape:", df_modern.shape)

Saved modern full parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_modern_full.parquet
Saved modern full csv    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\csv\idiomx_modern_full.csv
Shape: (14524, 64)


In [84]:
# [MERGE 3] Align both datasets to the union of columns

all_cols = sorted(main_cols | modern_cols)

for col in all_cols:
    if col not in df_main.columns:
        df_main[col] = pd.NA
    if col not in df_modern.columns:
        df_modern[col] = pd.NA

df_main = df_main[all_cols].copy()
df_modern = df_modern[all_cols].copy()

print("Aligned main shape  :", df_main.shape)
print("Aligned modern shape:", df_modern.shape)
print("Total unified columns:", len(all_cols))

Aligned main shape  : (174956, 63)
Aligned modern shape: (14524, 63)
Total unified columns: 63


In [63]:
# [MERGE 4] Concatenate both full datasets

df_extended_full = pd.concat([df_main, df_modern], ignore_index=True)

print("Merged full shape:", df_extended_full.shape)
print("\nSource dataset distribution:")
print(df_extended_full["source_dataset"].value_counts(dropna=False))

C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\3604072541.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_extended_full = pd.concat([df_main, df_modern], ignore_index=True)
C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\3604072541.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_extended_full = pd.concat([df_main, df_modern], ignore_index=True)


Merged full shape: (189480, 63)

Source dataset distribution:
source_dataset
idiomx_main         174956
modern_extension     14524
Name: count, dtype: int64


In [64]:
# [MERGE CHECK] Check duplicates after merge

dup_examples = df_extended_full.duplicated(subset=["example_normalized"]).sum()
dup_pairs = df_extended_full.duplicated(subset=["idiom_canonical", "example_normalized"]).sum()

print("Duplicate examples:", dup_examples)
print("Duplicate idiom-example pairs:", dup_pairs)

Duplicate examples: 2566
Duplicate idiom-example pairs: 2557


In [65]:
# [MERGE FIX] Remove duplicate idiom-example pairs (priority to main dataset)

# Step 1: prioritize main dataset
df_extended_full["source_priority"] = df_extended_full["source_dataset"].map({
    "idiomx_main": 0,
    "modern_extension": 1
})

# Step 2: sort so main comes first
df_extended_full = df_extended_full.sort_values(
    by=["idiom_canonical", "example_normalized", "source_priority"]
)

# Step 3: drop duplicates
before = len(df_extended_full)

df_extended_full = df_extended_full.drop_duplicates(
    subset=["idiom_canonical", "example_normalized"],
    keep="first"
)

after = len(df_extended_full)

print("Removed duplicates:", before - after)
print("New shape:", df_extended_full.shape)

# cleanup helper column
df_extended_full = df_extended_full.drop(columns=["source_priority"])

Removed duplicates: 2557
New shape: (186923, 64)


In [66]:
# [MERGE CHECK AGAIN]

dup_examples = df_extended_full.duplicated(subset=["example_normalized"]).sum()
dup_pairs = df_extended_full.duplicated(subset=["idiom_canonical", "example_normalized"]).sum()

print("Duplicate examples:", dup_examples)
print("Duplicate idiom-example pairs:", dup_pairs)

Duplicate examples: 9
Duplicate idiom-example pairs: 0


In [70]:
# [MERGE SAVE FIX 1] Safely normalize dtypes before parquet save

import pandas as pd
import numpy as np

def to_nullable_bool(series: pd.Series) -> pd.Series:
    """
    Convert mixed bool-like values to pandas nullable boolean dtype safely.
    """
    s = series.copy()

    def convert(x):
        if pd.isna(x):
            return pd.NA

        if isinstance(x, bool):
            return x

        text = str(x).strip().lower()

        if text in {"true", "1", "yes", "y"}:
            return True
        if text in {"false", "0", "no", "n"}:
            return False
        if text in {"", "nan", "none", "null", "<na>"}:
            return pd.NA

        return pd.NA

    return s.map(convert).astype("boolean")


# -----------------------------
# numeric columns
# -----------------------------
numeric_cols = [
    "idiom_confidence",
    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
]

for col in numeric_cols:
    if col in df_extended_full.columns:
        df_extended_full[col] = pd.to_numeric(df_extended_full[col], errors="coerce")

# -----------------------------
# boolean columns
# -----------------------------
bool_cols = [
    "is_example_idiom",
    "is_generated_example",
    "is_adversarial_example",
    "is_idiom",
]

for col in bool_cols:
    if col in df_extended_full.columns:
        df_extended_full[col] = to_nullable_bool(df_extended_full[col])

# -----------------------------
# string-like columns
# -----------------------------
for col in df_extended_full.columns:
    if col not in numeric_cols + bool_cols:
        df_extended_full[col] = df_extended_full[col].astype("string")

print("Dtype normalization completed.\n")
print(df_extended_full[bool_cols].dtypes)

Dtype normalization completed.

is_example_idiom          boolean
is_generated_example      boolean
is_adversarial_example    boolean
is_idiom                  boolean
dtype: object


In [71]:
# [MERGE SAVE FIX] Clean dtypes before saving parquet

import pandas as pd

# numeric columns
numeric_cols = [
    "idiom_confidence",
    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
]

for col in numeric_cols:
    if col in df_extended_full.columns:
        df_extended_full[col] = pd.to_numeric(df_extended_full[col], errors="coerce")

# boolean columns
bool_cols = [
    "is_example_idiom",
    "is_generated_example",
    "is_adversarial_example",
    "is_idiom",
]

for col in bool_cols:
    if col in df_extended_full.columns:
        df_extended_full[col] = df_extended_full[col].astype("boolean")

# keep text/object columns consistent
for col in df_extended_full.columns:
    if col not in numeric_cols + bool_cols:
        df_extended_full[col] = df_extended_full[col].astype("string")

print(df_extended_full.dtypes)

adversarial_type      string[python]
ambiguity_flag        string[python]
context_type          string[python]
enrichment_model      string[python]
enrichment_version    string[python]
                           ...      
source_style          string[python]
source_type           string[python]
source_url            string[python]
tags                  string[python]
validation_status     string[python]
Length: 63, dtype: object


In [43]:
# [MERGE 5 - OPTIONAL] Remove exact duplicate example rows only

dedup_keys = [
    "idiom_canonical",
    "example",
    "example_usage_label",
    "row_type",
    "context_type"
]

existing_dedup_keys = [c for c in dedup_keys if c in df_extended_full.columns]

before = len(df_extended_full)
df_extended_full = df_extended_full.drop_duplicates(subset=existing_dedup_keys).copy()
after = len(df_extended_full)

print(f"Rows before dedup: {before}")
print(f"Rows after dedup : {after}")
print(f"Removed          : {before - after}")

Rows before dedup: 189480
Rows after dedup : 189455
Removed          : 25


In [86]:
# [MERGE 6] Reorder columns for cleaner final export

preferred_order = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "example_usage_label",
    "is_example_idiom",

    "idiom_canonical_meaning",
    "idiom_canonical_meaning_arabic",
    "idiom_canonical_meaning_french",

    "idiom_in_example_meaning_en",
    "idiom_in_example_meaning_arabic",
    "idiom_in_example_meaning_french",

    "example_raw",
    "example_normalized",
    "example_language",
    "meaning_language",

    "source",
    "source_type",
    "source_url",
    "record_origin",
    "license_source",
    "source_dataset",

    "pos",
    "tags",
    "idiom_confidence",

    "is_idiom",
    "idiom_validity_label",
    "ambiguity_flag",
    "idiom_compositionality_level",
    "idiom_register",
    "idiom_domain",
    "learner_difficulty",
    "slang_strength",
    "regionality",
    "offensive_flag",

    "idiom_in_example_arabic",
    "idiom_in_example_french",

    "idiom_level_explanation_en",
    "idiom_level_explanation_ar",
    "idiom_level_explanation_fr",

    "explanation_en",
    "explanation_ar",
    "explanation_fr",

    "hard_negative_idioms",
    "meaning_paraphrases_en",
    "meaning_paraphrases_ar",
    "meaning_paraphrases_fr",

    "is_generated_example",
    "enrichment_model",
    "enrichment_version",
    "validation_status",

    "context_type",
    "source_style",
    "minimal_pair_id",
    "paraphrase_group_id",

    "is_adversarial_example",
    "adversarial_type",
    "expected_label",
    "row_type",

    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
    "semantic_quality",
]

first_cols = [c for c in preferred_order if c in df_extended_full.columns]
remaining_cols = [c for c in df_extended_full.columns if c not in first_cols]

df_extended_full = df_extended_full[first_cols + remaining_cols].copy()

print("Final merged columns:")
print(df_extended_full.columns.tolist())
print("\nFinal merged shape:", df_extended_full.shape)

Final merged columns:
['idiom_id', 'idiom_canonical', 'idiom_surface', 'example', 'example_usage_label', 'is_example_idiom', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_french', 'example_raw', 'example_normalized', 'example_language', 'meaning_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'source_dataset', 'pos', 'tags', 'idiom_confidence', 'is_idiom', 'idiom_validity_label', 'ambiguity_flag', 'idiom_compositionality_level', 'idiom_register', 'idiom_domain', 'learner_difficulty', 'slang_strength', 'regionality', 'offensive_flag', 'idiom_in_example_arabic', 'idiom_in_example_french', 'idiom_level_explanation_en', 'idiom_level_explanation_ar', 'idiom_level_explanation_fr', 'explanation_en', 'explanation_ar', 'explanation_fr', 'hard_negative_idioms', 'meaning_paraphrases_en', 'meaning_paraphrases_ar', 'meaning

In [87]:
# [MERGE 8] Compute and fill derived text-quality columns

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def normalize_example_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    # normalize quotes/apostrophes
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')

    # remove extra surrounding punctuation spacing
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    return text


def count_words(text: str) -> int:
    if pd.isna(text):
        return 0
    text = str(text).strip()
    if not text:
        return 0
    return len(re.findall(r"\S+", text))


def compute_tfidf_similarity_batch(examples, meanings):
    """
    Lightweight semantic similarity using TF-IDF cosine similarity.
    Good enough as a reproducible fallback for dataset-level scoring.
    """
    sims = []

    for ex, mean in zip(examples, meanings):
        ex = "" if pd.isna(ex) else str(ex).strip()
        mean = "" if pd.isna(mean) else str(mean).strip()

        if not ex or not mean:
            sims.append(np.nan)
            continue

        try:
            vect = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
            X = vect.fit_transform([ex, mean])
            sim = cosine_similarity(X[0:1], X[1:2])[0][0]
            sims.append(float(round(sim, 4)))
        except Exception:
            sims.append(np.nan)

    return sims


def semantic_quality_from_score(score):
    if pd.isna(score):
        return pd.NA
    if score >= 0.35:
        return "high"
    elif score >= 0.18:
        return "medium"
    else:
        return "low"


# --------------------------------------------------
# Ensure columns exist
# --------------------------------------------------
target_cols = [
    "example_normalized",
    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
    "semantic_quality",
]

for col in target_cols:
    if col not in df_extended_full.columns:
        df_extended_full[col] = pd.NA

# --------------------------------------------------
# Fill example_normalized
# --------------------------------------------------
mask_norm = df_extended_full["example_normalized"].isna() | (df_extended_full["example_normalized"].astype(str).str.strip() == "")
df_extended_full.loc[mask_norm, "example_normalized"] = (
    df_extended_full.loc[mask_norm, "example"].apply(normalize_example_text)
)

# --------------------------------------------------
# Fill sentence length columns
# --------------------------------------------------
mask_chars = df_extended_full["sentence_length_chars"].isna()
df_extended_full.loc[mask_chars, "sentence_length_chars"] = (
    df_extended_full.loc[mask_chars, "example_normalized"].fillna("").astype(str).str.len()
)

mask_words = df_extended_full["sentence_length_words"].isna()
df_extended_full.loc[mask_words, "sentence_length_words"] = (
    df_extended_full.loc[mask_words, "example_normalized"].apply(count_words)
)

# --------------------------------------------------
# Fill semantic similarity
# Uses:
#   example  vs idiom_in_example_meaning_en
# --------------------------------------------------
mask_sim = df_extended_full["semantic_similarity_example_vs_meaning"].isna()

examples = df_extended_full.loc[mask_sim, "example"].tolist()
meanings = df_extended_full.loc[mask_sim, "idiom_in_example_meaning_en"].tolist()

df_extended_full.loc[mask_sim, "semantic_similarity_example_vs_meaning"] = compute_tfidf_similarity_batch(
    examples,
    meanings
)

# --------------------------------------------------
# Fill semantic_quality from similarity
# --------------------------------------------------
mask_quality = df_extended_full["semantic_quality"].isna() | (df_extended_full["semantic_quality"].astype(str).str.strip() == "")
df_extended_full.loc[mask_quality, "semantic_quality"] = (
    df_extended_full.loc[mask_quality, "semantic_similarity_example_vs_meaning"]
    .apply(semantic_quality_from_score)
)

# --------------------------------------------------
# Optional dtype cleanup
# --------------------------------------------------
df_extended_full["sentence_length_chars"] = pd.to_numeric(
    df_extended_full["sentence_length_chars"], errors="coerce"
).astype("Int64")

df_extended_full["sentence_length_words"] = pd.to_numeric(
    df_extended_full["sentence_length_words"], errors="coerce"
).astype("Int64")

df_extended_full["semantic_similarity_example_vs_meaning"] = pd.to_numeric(
    df_extended_full["semantic_similarity_example_vs_meaning"], errors="coerce"
)

print("Derived columns updated.\n")
print(df_extended_full[
    [
        "example",
        "example_normalized",
        "sentence_length_chars",
        "sentence_length_words",
        "idiom_in_example_meaning_en",
        "semantic_similarity_example_vs_meaning",
        "semantic_quality",
    ]
].head(10))

C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\1003336152.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<StringArray>
[]
Length: 0, dtype: string' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_extended_full.loc[mask_words, "sentence_length_words"] = (


Derived columns updated.

                                              example  \
10  ’Ark at ’ee, as if you could ever finish on time!   
8   ’Ark at ’ee! What do you mean by quitting so s...   
12   'ark at 'ee, you don’t really mean that, do you?   
0   'ark at 'ee, you really think that plan will w...   
2   As the storm worsened, Togget shouted, ‘’Ark a...   
7   Can you ’ark at ’ee how clear the sound is fro...   
9   Can you ’ark at ’ee the diagram to find the au...   
1   Can you 'ark at 'ee to check if the noise is c...   
13  He asked me to ’ark at ’ee while he fixed the ...   
4   In the meeting, he remarked, ‘’Ark at ’ee, how...   

                                   example_normalized  sentence_length_chars  \
10      ark at ee as if you could ever finish on time                     49   
8   ark at ee what do you mean by quitting so sudd...                     54   
12         ark at ee you dont really mean that do you                     48   
0      ark at ee you reall

In [88]:
df_extended_full["is_example_idiom"] = df_extended_full["example_usage_label"].map({
    "idiomatic": True,
    "literal": False,
    "borderline": False
})

In [89]:
# [MERGE SAVE FIX 2] Save final extended dataset

publication_dir = BASE_DIR / "data" / "final" / "publication"
publication_csv_dir = publication_dir / "csv"

publication_dir.mkdir(parents=True, exist_ok=True)
publication_csv_dir.mkdir(parents=True, exist_ok=True)

extended_full_parquet = publication_dir / "idiomx_extended_full.parquet"
extended_full_csv = publication_csv_dir / "idiomx_extended_full.csv"

df_extended_full.to_parquet(extended_full_parquet, index=False)
df_extended_full.to_csv(extended_full_csv, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", extended_full_parquet)
print("-", extended_full_csv)
print("\nFinal shape:", df_extended_full.shape)
print("Unique idioms:", df_extended_full["idiom_canonical"].nunique())
print("Source dataset distribution:")
print(df_extended_full["source_dataset"].value_counts(dropna=False))

Saved:
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_full.parquet
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\csv\idiomx_extended_full.csv

Final shape: (186923, 63)
Unique idioms: 14054
Source dataset distribution:
source_dataset
idiomx_main         172403
modern_extension     14520
Name: count, dtype: Int64


In [7]:
# [MERGE 7] Save merged dataset

output_parquet.parent.mkdir(parents=True, exist_ok=True)
output_csv.parent.mkdir(parents=True, exist_ok=True)

df_extended_full.to_parquet(output_parquet, index=False)
df_extended_full.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("Saved parquet:", output_parquet)
print("Saved csv    :", output_csv)

Saved parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_full.parquet
Saved csv    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\csv\idiomx_extended_full.csv


In [90]:
# [MERGE 10] Train/test split by idiom_canonical and save as parquet

from pathlib import Path
from sklearn.model_selection import train_test_split

BASE_DIR = Path(r"C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset")

publication_dir = BASE_DIR / "data" / "final" / "publication"
publication_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Split by idiom to avoid leakage
# --------------------------------------------------
unique_idioms = (
    df_extended_full["idiom_canonical"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

train_idioms, test_idioms = train_test_split(
    unique_idioms,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

train_idiom_set = set(train_idioms)
test_idiom_set = set(test_idioms)

df_extended_train = df_extended_full[
    df_extended_full["idiom_canonical"].astype(str).str.strip().isin(train_idiom_set)
].copy()

df_extended_test = df_extended_full[
    df_extended_full["idiom_canonical"].astype(str).str.strip().isin(test_idiom_set)
].copy()

print("Extended full shape :", df_extended_full.shape)
print("Extended train shape:", df_extended_train.shape)
print("Extended test shape :", df_extended_test.shape)

print("\nUnique idioms:")
print("Train:", df_extended_train["idiom_canonical"].nunique())
print("Test :", df_extended_test["idiom_canonical"].nunique())

# safety check
overlap = set(df_extended_train["idiom_canonical"].dropna().astype(str).str.strip()) & \
          set(df_extended_test["idiom_canonical"].dropna().astype(str).str.strip())

print("\nOverlap idioms between train/test:", len(overlap))

Extended full shape : (186923, 63)
Extended train shape: (149174, 63)
Extended test shape : (37441, 63)

Unique idioms:
Train: 11243
Test : 2811

Overlap idioms between train/test: 0


In [91]:
# [MERGE 11] Save full + split parquet files

extended_full_parquet = publication_dir / "idiomx_extended_full.parquet"
extended_train_parquet = publication_dir / "idiomx_extended_train.parquet"
extended_test_parquet = publication_dir / "idiomx_extended_test.parquet"

df_extended_full.to_parquet(extended_full_parquet, index=False)
df_extended_train.to_parquet(extended_train_parquet, index=False)
df_extended_test.to_parquet(extended_test_parquet, index=False)

print("Saved parquet files:")
print("-", extended_full_parquet)
print("-", extended_train_parquet)
print("-", extended_test_parquet)

Saved parquet files:
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_full.parquet
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_train.parquet
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_test.parquet
